In [18]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

In [19]:
import os
os.listdir("/kaggle/input/datasets/arnavsinghal0906/amazon-reviews")

['Amazon_Reviews.csv']

In [20]:
import pandas as pd

df = pd.read_csv(
    "/kaggle/input/datasets/arnavsinghal0906/amazon-reviews/Amazon_Reviews.csv",
    engine="python",
    on_bad_lines="skip"
)

df = df[["Review Text", "Rating"]]

df["Rating"] = df["Rating"].astype(str).str.extract(r'(\d+)')

df = df.dropna(subset=["Rating"])

df["Rating"] = df["Rating"].astype(int)

df = df[df["Rating"] != 3]

df["label"] = df["Rating"].apply(lambda r: 1 if r >= 4 else 0)

texts = df["Review Text"].astype(str).tolist()
labels = df["label"].tolist()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
max_length = 128

class AmazonDataset(Dataset):

    def __init__(self, txt, lbl):
        self.txt = txt
        self.lbl = lbl

    def __len__(self):
        return len(self.txt)

    def __getitem__(self, idx):

        enc = tokenizer(
            self.txt[idx],
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(),
            "label": torch.tensor(self.lbl[idx])
        }

dataset = AmazonDataset(texts, labels)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [22]:
class GRUModel(nn.Module):

    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):

        x = self.embedding(x)

        _, hidden = self.gru(x)

        hidden = hidden.squeeze(0)

        out = self.fc(hidden)

        return out


model = GRUModel(tokenizer.vocab_size, 128, 256)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(3):

    epoch_loss = 0

    for batch in loader:

        inputs = batch["input_ids"]
        targets = batch["label"]

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = loss_fn(outputs, targets)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    print("Epoch:", epoch + 1, "Loss:", epoch_loss / len(loader))

Epoch: 1 Loss: 0.37439057338988724
Epoch: 2 Loss: 0.17707117881608178
Epoch: 3 Loss: 0.11732838731391942


In [24]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in loader:

        inputs = batch["input_ids"]
        targets = batch["label"]

        outputs = model(inputs)

        preds = torch.argmax(outputs, dim=1)

        correct += (preds == targets).sum().item()

        total += targets.size(0)

accuracy = correct / total

print("Accuracy:", accuracy)

Accuracy: 0.9737233515121467
